# Lawgic Taxonomy Fusion

**Objective:** build one combined, auditable dataset from **100 ToS**, **ToS;DR**, and **CLAUDETTE Cross Market** for later Lawgic model finetuning.

This notebook intentionally does **not** train a model. It prepares two dataset views:

1. **Long format**: one row per `(text, source annotation, Lawgic topic, score)`.
2. **Wide format**: one row per normalized text with 45-length `labels_presence`, `mask`, and `scores` vectors.

The most important invariant is:

> **A missing annotation is unknown, not negative.**
>
> If CLAUDETTE did not annotate privacy topics, that does not mean a clause is negative for every privacy head. Those dimensions must have `mask=0`, so the loss ignores them.

All cells are written so they can be inspected before execution. This notebook should be run manually after reviewing the mapping policies below.

## Pipeline Overview

```mermaid
flowchart TD
    Taxonomy["lawgic_topics.json"] --> MappingRules["Explicit mapping rules"]
    ClaudetteCSV["CLAUDETTE normalized CSV"] --> LongRecords["Long records"]
    TosdrCSV["ToS;DR points CSV"] --> LongRecords
    HundredTosCSV["100 ToS annotated comments CSV"] --> LongRecords
    MappingRules --> LongRecords
    LongRecords --> ConflictReview["Conflict review CSV"]
    LongRecords --> WideRecords["Wide records with labels and mask"]
    WideRecords --> TrainingData["Masked training input"]
```

The taxonomy file provides the list of valid Lawgic topic IDs and source mappings. This notebook adds one more layer: **resolution rules** for coarse native labels that map to multiple finer Lawgic topics.

Examples:

- CLAUDETTE `ltd` is kept on `limitation_of_liability` only. It is too coarse to safely label `liability_cap`, `warranty_disclaimer`, and `indemnification` as positives.
- CLAUDETTE `a` maps to both `mandatory_arbitration` and `class_action_waiver`, because that native unfairness category explicitly covers arbitration and class waivers together.
- 100 ToS codes stay fine-grained. They are the best signal for narrow heads like `ltd_cap`, `as_is`, `sever`, and `interpret`.

In [12]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until the Lawgic repository root is found."""
    start = (start or Path.cwd()).resolve()
    marker = Path("generated_files/lawgic_taxonomy/lawgic_topics.json")
    for candidate in (start, *start.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root containing {marker}")


PROJECT_ROOT = find_project_root()

TAXONOMY_PATH = PROJECT_ROOT / "generated_files/lawgic_taxonomy/lawgic_topics.json"
CLAUDETTE_CSV_PATH = PROJECT_ROOT / "generated_files/claudette_cross_market/claudette_cross_market_clauses.csv"
TOSDR_CSV_PATH = PROJECT_ROOT / "generated_files/tos_dr/tos_dr_points.csv"
HUNDRED_TOS_CSV_PATH = PROJECT_ROOT / "generated_files/100_tos/annotated_tos_comments.csv"
HUNDRED_TOS_LEGACY_CSV_PATH = PROJECT_ROOT / "generated_files/100_tos/cleaned_tos_comments.csv"
HUNDRED_TOS_VARIABLES_PATH = PROJECT_ROOT / "generated_files/100_tos/100_tos_eval_variables.json"

OUTPUT_DIR = PROJECT_ROOT / "generated_files/lawgic_taxonomy"
LONG_OUTPUT_PATH = OUTPUT_DIR / "lawgic_combined_long.csv"
WIDE_OUTPUT_PATH = OUTPUT_DIR / "lawgic_combined_wide.csv"
CONFLICTS_OUTPUT_PATH = OUTPUT_DIR / "lawgic_combined_conflicts.csv"
HUNDRED_TOS_DIAGNOSTICS_OUTPUT_PATH = OUTPUT_DIR / "lawgic_100_tos_parse_diagnostics.csv"
TOSDR_DIAGNOSTICS_OUTPUT_PATH = OUTPUT_DIR / "lawgic_tosdr_parse_diagnostics.csv"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "lawgic_fusion_summary.json"

# ToS;DR has a non-rubric class called "blocker". We treat it as harmful because
# it marks a severe obstacle to user rights in the ToS;DR data model.
TOSDR_SCORE_MAP = {
    "bad": -1,
    "blocker": -1,
    "neutral": 0,
    "good": 1,
}

# Current training recommendation: use the binary presence target with masked BCE.
# The harm scores are still carried forward for the future score head.
PRESENCE_LABEL_FOR_ANNOTATED_TOPIC = 1.0

pd.set_option("display.max_colwidth", 140)
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/riki/Coding Projects/Thesis/lawgic


## Taxonomy Loading and Explicit Mapping Policy

`lawgic_topics.json` is the master topic vocabulary. This notebook does **not** invent topics. Every mapped label must resolve to one of the 45 topic IDs in that file.

The raw `source_mappings` inside the taxonomy are intentionally broad in places. For fusion, broad labels need an explicit policy:

| Source label | Fusion decision | Reason |
| --- | --- | --- |
| CLAUDETTE `ltd` | `limitation_of_liability` only | Native label is too coarse to safely mark liability caps, warranty disclaimers, and indemnity as positives. |
| CLAUDETTE `a` | `mandatory_arbitration` + `class_action_waiver` | Native category jointly covers arbitration and class waivers. |
| CLAUDETTE `ch` | `contract_changes` only | Native category is broad unilateral change; 100 ToS gives finer subtype labels. |
| CLAUDETTE `ter` | `account_termination` | Closest fit to unilateral termination. |
| CLAUDETTE `cr` | `content_removal` | Native category means content removal/censorship, not all content rules. |
| ToS;DR `Guarantee` | `warranty_disclaimer` | Avoid fanning a generic topic into all limitation-remedy children. |
| ToS;DR `Waivers` | `indemnification` | Use narrowest stable fit for generic waiver/hold-harmless language. |
| 100 ToS codes | direct mapping | 100 ToS is already fine-grained and score-rubric based. |

In [13]:
with TAXONOMY_PATH.open("r", encoding="utf-8") as f:
    taxonomy = json.load(f)

topics = taxonomy["topics"]
topic_ids = [topic["id"] for topic in topics]
topic_id_to_index = {topic_id: idx for idx, topic_id in enumerate(topic_ids)}
valid_topic_ids = set(topic_ids)

if len(topic_ids) != len(valid_topic_ids):
    duplicates = [topic_id for topic_id, count in Counter(topic_ids).items() if count > 1]
    raise ValueError(f"Duplicate Lawgic topic IDs: {duplicates}")

if len(topic_ids) != 45:
    raise ValueError(f"Expected 45 Lawgic topics, found {len(topic_ids)}")

# Raw source mapping lookup from the taxonomy. This is used for direct mappings
# and for validation, but ambiguous source labels are overridden below.
raw_source_to_topics: dict[str, dict[str, list[str]]] = defaultdict(lambda: defaultdict(list))
for topic in topics:
    for source_name, native_labels in topic.get("source_mappings", {}).items():
        for native_label in native_labels:
            raw_source_to_topics[source_name][native_label].append(topic["id"])

CLAUDETTE_TOPIC_RULES = {
    "law": ["choice_of_law"],
    "j": ["choice_of_forum"],
    "a": ["mandatory_arbitration", "class_action_waiver"],
    "ltd": ["limitation_of_liability"],
    "ch": ["contract_changes"],
    "ter": ["account_termination"],
    "cr": ["content_removal"],
    "use": ["contract_by_use"],
    "pinc": ["privacy_incorporation"],
}

# ToS;DR overrides prevent generic topic fan-out where the raw taxonomy mapping
# would mark too many fine-grained heads as positive from one broad label.
TOSDR_TOPIC_OVERRIDES = {
    "Jurisdiction and governing laws": ["choice_of_law", "choice_of_forum"],
    "Dispute Resolution": ["mandatory_arbitration", "class_action_waiver"],
    "Guarantee": ["warranty_disclaimer"],
    "Waivers": ["indemnification"],
    "Changes": ["contract_changes"],
}

HUNDRED_TOS_CODE_ALIASES = {
    "ip": "IP",
    "tran": "transfer",
}

MAPPING_NOTES = {
    "claudette:ltd": "Mapped only to limitation_of_liability to avoid inventing subtype positives for cap, warranty, or indemnity.",
    "claudette:a": "Mapped to both arbitration and class action waiver because CLAUDETTE groups those together.",
    "claudette:ch": "Mapped only to contract_changes because CLAUDETTE does not distinguish contract, service, price, notice, and user-involvement changes.",
    "tos_dr:Guarantee": "Mapped to warranty_disclaimer as the narrowest stable limitation-remedy interpretation.",
    "tos_dr:Waivers": "Mapped to indemnification rather than every waiver-adjacent topic.",
    "100_tos": "100 ToS eval variables map directly because they are already fine-grained.",
}


def assert_valid_topic_rules(rule_name: str, rules: dict[str, list[str]]) -> None:
    """Fail early if a hand-written mapping points to a non-existent topic."""
    invalid = sorted({topic_id for ids in rules.values() for topic_id in ids if topic_id not in valid_topic_ids})
    if invalid:
        raise ValueError(f"{rule_name} contains unknown Lawgic topic IDs: {invalid}")


assert_valid_topic_rules("CLAUDETTE_TOPIC_RULES", CLAUDETTE_TOPIC_RULES)
assert_valid_topic_rules("TOSDR_TOPIC_OVERRIDES", TOSDR_TOPIC_OVERRIDES)

print(f"Loaded {len(topic_ids)} Lawgic topics")

Loaded 45 Lawgic topics


## Shared Helpers and Long-Record Schema

Every source parser emits the same long-record schema. The wide training table is derived only from these long records.

Important columns:

- `text`: original clause text used for display.
- `normalized_text`: NFKC-normalized, whitespace-collapsed key used for grouping.
- `lawgic_topic_id`: one of the 45 taxonomy IDs.
- `mapped_score`: consumer-harm score in `{-1, 0, 1}`.
- `presence_label`: always `1.0` for a parsed positive topic annotation.
- `mask`: not stored in long format; derived later in wide format.
- `metadata_json`: source-specific details kept for audit without exploding the common schema.

A row exists only when a source explicitly labeled that text with that topic. Absence of a row is not negative evidence.

In [14]:
def normalize_text(value: Any) -> str:
    """Normalize clause text for grouping without destroying display text.

    NFKC handles compatibility glyphs such as ligatures. Collapsing whitespace
    makes embedded newlines and repeated spaces comparable across source CSVs.
    """
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def compact_json(value: Any) -> str:
    """Serialize metadata consistently for CSV output."""
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def make_long_record(
    *,
    text: Any,
    source_dataset: str,
    source_id: str,
    lawgic_topic_id: str,
    mapped_score: int,
    native_label: str,
    native_tag: str | None = None,
    native_score: Any = None,
    service_name: str | None = None,
    platform: str | None = None,
    company: str | None = None,
    parse_status: str = "parsed",
    mapping_rule: str = "direct",
    metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Create one canonical long-format annotation record."""
    normalized_text = normalize_text(text)
    if not normalized_text:
        raise ValueError(f"Empty normalized text for {source_dataset}:{source_id}")
    if lawgic_topic_id not in valid_topic_ids:
        raise ValueError(f"Unknown Lawgic topic ID: {lawgic_topic_id}")
    if mapped_score not in {-1, 0, 1}:
        raise ValueError(f"Invalid mapped_score {mapped_score} for {source_dataset}:{source_id}")

    return {
        "text": str(text),
        "normalized_text": normalized_text,
        "source_dataset": source_dataset,
        "source_id": str(source_id),
        "service_name": service_name,
        "platform": platform,
        "company": company,
        "lawgic_topic_id": lawgic_topic_id,
        "topic_index": topic_id_to_index[lawgic_topic_id],
        "mapped_score": int(mapped_score),
        "presence_label": PRESENCE_LABEL_FOR_ANNOTATED_TOPIC,
        "native_label": native_label,
        "native_tag": native_tag,
        "native_score": native_score,
        "parse_status": parse_status,
        "mapping_rule": mapping_rule,
        "metadata_json": compact_json(metadata or {}),
    }


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str:
    """Return the first column that exists, with a clear error if none do."""
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"None of these columns exist: {candidates}. Available columns: {list(df.columns)}")

## Parse CLAUDETTE Cross Market

CLAUDETTE has already been normalized into `claudette_cross_market_clauses.csv` by the prior notebook.

Policy here:

- Use `quoted_text` as the clause text.
- Use `claudette_topic_code` for topic mapping.
- Use existing `mapped_score` (`1`, `0`, `-1`) and **not** `binary_label`.
- Expand one source row to multiple long records only when the mapping policy says the native label truly covers multiple Lawgic topics. Currently that is CLAUDETTE `a` → arbitration + class action waiver.

In [15]:
def parse_claudette() -> pd.DataFrame:
    """Parse CLAUDETTE rows into Lawgic long records."""
    df = pd.read_csv(CLAUDETTE_CSV_PATH)
    records: list[dict[str, Any]] = []

    for row in df.itertuples(index=False):
        topic_code = getattr(row, "claudette_topic_code")
        topic_ids_for_code = CLAUDETTE_TOPIC_RULES.get(topic_code)
        if not topic_ids_for_code:
            raise ValueError(f"No CLAUDETTE mapping rule for topic code: {topic_code}")

        source_id = f"{getattr(row, 'platform')}:{getattr(row, 'sentence_index')}:{getattr(row, 'claudette_tag')}"
        mapping_rule = f"claudette:{topic_code}"
        metadata = {
            "claudette_fairness_level": getattr(row, "claudette_fairness_level"),
            "claudette_fairness_label": getattr(row, "claudette_fairness_label"),
            "binary_label": getattr(row, "binary_label"),
        }

        for topic_id in topic_ids_for_code:
            records.append(
                make_long_record(
                    text=getattr(row, "quoted_text"),
                    source_dataset="claudette",
                    source_id=source_id,
                    platform=getattr(row, "platform"),
                    lawgic_topic_id=topic_id,
                    mapped_score=int(getattr(row, "mapped_score")),
                    native_label=topic_code,
                    native_tag=getattr(row, "claudette_tag"),
                    native_score=getattr(row, "claudette_fairness_level"),
                    mapping_rule=mapping_rule,
                    metadata=metadata,
                )
            )

    return pd.DataFrame.from_records(records)


claudette_long = parse_claudette()
claudette_long.head()

,text,normalized_text,source_dataset,source_id,service_name,platform,company,lawgic_topic_id,topic_index,mapped_score,presence_label,native_label,native_tag,native_score,parse_status,mapping_rule,metadata_json
0,"By using the Service, you agree to these TOS.","By using the Service, you agree to these TOS.",claudette,23andme:36:use2,None,23andme,None,contract_by_use,17,0,1.0,use,use2,2,parsed,claudette:use,"{""binary_label"": 1, ""claudette_fairness_label"": ""potentially unfair"", ""claudette_fairness_level"": 2}"
1,You therefore acknowledge and agree that the form and nature of the Services which 23andMe provides may change from time to time.,You therefore acknowledge and agree that the form and nature of the Services which 23andMe provides may change from time to time.,claudette,23andme:87:ch2,None,23andme,None,contract_changes,9,0,1.0,ch,ch2,2,parsed,claudette:ch,"{""binary_label"": 1, ""claudette_fairness_label"": ""potentially unfair"", ""claudette_fairness_level"": 2}"
2,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) providing some Servi...","As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) providing some Servi...",claudette,23andme:88:ch2,None,23andme,None,contract_changes,9,0,1.0,ch,ch2,2,parsed,claudette:ch,"{""binary_label"": 1, ""claudette_fairness_label"": ""potentially unfair"", ""claudette_fairness_level"": 2}"
3,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) providing some Servi...","As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) providing some Servi...",claudette,23andme:88:ter3,None,23andme,None,account_termination,14,-1,1.0,ter,ter3,3,parsed,claudette:ter,"{""binary_label"": 1, ""claudette_fairness_label"": ""clearly unfair"", ""claudette_fairness_level"": 3}"
4,You acknowledge and agree that while 23andMe may not currently have set a fixed upper limit on the number of transmissions you may send ...,You acknowledge and agree that while 23andMe may not currently have set a fixed upper limit on the number of transmissions you may send ...,claudette,23andme:96:ch2,None,23andme,None,contract_changes,9,0,1.0,ch,ch2,2,parsed,claudette:ch,"{""binary_label"": 1, ""claudette_fairness_label"": ""potentially unfair"", ""claudette_fairness_level"": 2}"


## Parse ToS;DR

ToS;DR contributes broad community-maintained point topics and point classifications.

Policy here:

- Use `point_quote_text` as the clause text.
- Use `cases.case_topic` as the native topic label.
- Map `cases.case_classification` to Lawgic score: `bad=-1`, `blocker=-1`, `neutral=0`, `good=1`.
- Use explicit overrides for broad topics that would otherwise fan out too widely through `source_mappings`.
- Preserve `point_id`, `service_name`, `score_rubric`, and classification metadata for later audit.

`blocker` is treated as `-1` because it marks a severe user-rights obstacle in ToS;DR. The row keeps `parse_status="blocker_as_bad"` so this policy can be audited later.

In [16]:
def map_tosdr_topic(native_topic: str) -> list[str]:
    """Map a ToS;DR topic to Lawgic topic IDs.

    Overrides are checked first. If no override exists, fall back to the taxonomy's
    `source_mappings`. This keeps most taxonomy maintenance in `lawgic_topics.json`
    while making high-risk broad mappings explicit here.
    """
    if native_topic in TOSDR_TOPIC_OVERRIDES:
        return TOSDR_TOPIC_OVERRIDES[native_topic]
    return raw_source_to_topics["tos_dr"].get(native_topic, [])


def parse_tosdr() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Parse ToS;DR point rows into Lawgic long records plus diagnostics.

    Some ToS;DR rows have a topic/classification but no quote text. Those rows
    cannot become training examples because they would produce empty model input.
    We skip them and write diagnostics instead of crashing or silently dropping.
    """
    df = pd.read_csv(TOSDR_CSV_PATH)

    text_col = first_existing_column(df, ["point_quote_text", "quote_text", "text"])
    topic_col = first_existing_column(df, ["cases.case_topic", "case_topic"])
    class_col = first_existing_column(df, ["cases.case_classification", "case_classification"])
    point_id_col = first_existing_column(df, ["point_id", "id"])
    service_col = "service_name" if "service_name" in df.columns else None
    rubric_col = "score_rubric" if "score_rubric" in df.columns else None

    records: list[dict[str, Any]] = []
    diagnostics: list[dict[str, Any]] = []
    unmapped_topics = Counter()

    # Use dictionaries instead of itertuples because ToS;DR column names contain
    # dots (`cases.case_topic`) that are awkward as Python attribute names.
    for row_index, row in enumerate(df.to_dict("records")):
        native_topic = row[topic_col]
        native_class = row[class_col]
        normalized_class = str(native_class).strip().lower()
        point_id = row[point_id_col]
        service_name = row[service_col] if service_col else None
        quote_text = row[text_col]

        if not normalize_text(quote_text):
            diagnostics.append({
                "row_index": row_index,
                "point_id": point_id,
                "service_name": service_name,
                "native_topic": native_topic,
                "native_class": native_class,
                "reason": "empty_point_quote_text",
            })
            continue

        if normalized_class not in TOSDR_SCORE_MAP:
            raise ValueError(f"Unexpected ToS;DR classification: {native_class}")

        topic_ids_for_label = map_tosdr_topic(str(native_topic))
        if not topic_ids_for_label:
            unmapped_topics[str(native_topic)] += 1
            diagnostics.append({
                "row_index": row_index,
                "point_id": point_id,
                "service_name": service_name,
                "native_topic": native_topic,
                "native_class": native_class,
                "reason": "unmapped_tosdr_topic",
            })
            continue

        score_rubric = row[rubric_col] if rubric_col else None
        parse_status = "blocker_as_bad" if normalized_class == "blocker" else "parsed"
        metadata = {
            "point_id": point_id,
            "case_classification": native_class,
            "score_rubric": score_rubric,
        }

        for topic_id in topic_ids_for_label:
            records.append(
                make_long_record(
                    text=quote_text,
                    source_dataset="tos_dr",
                    source_id=str(point_id),
                    service_name=service_name,
                    lawgic_topic_id=topic_id,
                    mapped_score=TOSDR_SCORE_MAP[normalized_class],
                    native_label=str(native_topic),
                    native_score=native_class,
                    parse_status=parse_status,
                    mapping_rule=f"tos_dr:{native_topic}",
                    metadata=metadata,
                )
            )

    if unmapped_topics:
        print("Unmapped ToS;DR topics (kept out of long records):")
        print(unmapped_topics)

    return pd.DataFrame.from_records(records), pd.DataFrame.from_records(diagnostics)


tosdr_long, tosdr_diagnostics = parse_tosdr()
tosdr_long.head()

,text,normalized_text,source_dataset,source_id,service_name,platform,company,lawgic_topic_id,topic_index,mapped_score,presence_label,native_label,native_tag,native_score,parse_status,mapping_rule,metadata_json
0,<li>what we may use your personal data for;,<li>what we may use your personal data for;,tos_dr,17470,Telegram,None,None,recommender_transparency,35,1,1.0,Transparency,None,good,parsed,tos_dr:Transparency,"{""case_classification"": ""good"", ""point_id"": 17470, ""score_rubric"": ""-1: The terms and policies are inaccessible, unclear, or not drafted..."
1,<li>what we may use your personal data for;,<li>what we may use your personal data for;,tos_dr,17470,Telegram,None,None,interpretation_clause,37,1,1.0,Transparency,None,good,parsed,tos_dr:Transparency,"{""case_classification"": ""good"", ""point_id"": 17470, ""score_rubric"": ""-1: The terms and policies are inaccessible, unclear, or not drafted..."
2,<li>what we may use your personal data for;,<li>what we may use your personal data for;,tos_dr,17470,Telegram,None,None,transparency,41,1,1.0,Transparency,None,good,parsed,tos_dr:Transparency,"{""case_classification"": ""good"", ""point_id"": 17470, ""score_rubric"": ""-1: The terms and policies are inaccessible, unclear, or not drafted..."
3,We do not use cookies for profiling or advertising.,We do not use cookies for profiling or advertising.,tos_dr,17474,Telegram,None,None,third_parties,29,1,1.0,Third Parties,None,good,parsed,tos_dr:Third Parties,"{""case_classification"": ""good"", ""point_id"": 17474, ""score_rubric"": ""-1: Personal data is shared with, or sold to, third parties without ..."
4,"Deleting your account removes all messages, media, contacts and every other piece of data you store in the Telegram cloud.\r\nThis actio...","Deleting your account removes all messages, media, contacts and every other piece of data you store in the Telegram cloud. This action m...",tos_dr,17480,Telegram,None,None,content_retrieval,21,1,1.0,Right to Leave The Service,None,good,parsed,tos_dr:Right to Leave The Service,"{""case_classification"": ""good"", ""point_id"": 17480, ""score_rubric"": ""-1: The user cannot freely terminate, or cannot delete or retrieve t..."


## Parse 100 ToS

100 ToS is the messiest source because the `comment` field mixes structured eval variables with documentary notes and free text.

Preferred input is `annotated_tos_comments.csv` rather than `cleaned_tos_comments.csv` because it preserves semicolon-separated multi-annotations such as:

- `acc_sus 0; acc_del 0`
- `cnt_del 0; cnt_modr 0`
- `ip -1; shad_ip`

Training policy:

- Accept only structured `code score` annotations where `score ∈ {-1, 0, 1}`.
- Split semicolon/comma separated comments into candidate segments.
- Validate every code against `100_tos_eval_variables.json` after alias normalization.
- Exclude `docu`, `uncle`, free text, unknown custom notes, and explanatory legal tails.
- Skip rows with empty `referenced_text` even when the comment parses cleanly (for example `Google.docx,8` has `ip 0` but no clause text).
- Log every excluded segment and every skipped empty-text row to `lawgic_100_tos_parse_diagnostics.csv` when output cells are run.

This is deliberately conservative. It is better to keep ambiguous 100 ToS rows out of training than to create noisy labels.

In [17]:
with HUNDRED_TOS_VARIABLES_PATH.open("r", encoding="utf-8") as f:
    hundred_tos_variables = json.load(f)

HUNDRED_TOS_CODES = {item["code"] for item in hundred_tos_variables}

# Build direct 100 ToS code -> Lawgic topic mapping from the taxonomy. Some codes
# map to more than one topic in the taxonomy (`IP` maps to copyright_license and
# ownership through source_mappings). That is allowed only where the taxonomy says so.
HUNDRED_TOS_CODE_TO_TOPICS = {
    code: raw_source_to_topics["100_tos"].get(code, []) for code in HUNDRED_TOS_CODES
}

missing_hundred_tos_mappings = {
    code: topics_for_code for code, topics_for_code in HUNDRED_TOS_CODE_TO_TOPICS.items() if not topics_for_code
}
if missing_hundred_tos_mappings:
    raise ValueError(f"100 ToS codes missing Lawgic mappings: {missing_hundred_tos_mappings}")


def canonicalize_hundred_tos_code(raw_code: str) -> str:
    """Normalize known spelling/case variants in 100 ToS comments."""
    cleaned = raw_code.strip()
    return HUNDRED_TOS_CODE_ALIASES.get(cleaned, cleaned)


def classify_unparsed_100_tos_segment(segment: str) -> str:
    """Classify excluded 100 ToS segments for diagnostics."""
    lowered = segment.strip().lower()
    if not lowered:
        return "empty_segment"
    if lowered.startswith("docu"):
        return "documentary_note"
    if lowered.startswith("uncle"):
        return "unclear_or_legal_savings_fragment"
    first_token = lowered.split(maxsplit=1)[0]
    canonical = canonicalize_hundred_tos_code(first_token)
    if canonical not in HUNDRED_TOS_CODES:
        return "unknown_code_or_free_text"
    return "known_code_without_valid_score"


def split_100_tos_comment(comment: Any) -> list[str]:
    """Split comments into parse candidates.

    Semicolons are the main separator in `annotated_tos_comments.csv`. Commas are
    also accepted for legacy rows like `acc_sus 0, acc_del 0`. Splitting is still
    conservative because each resulting segment must independently match `code score`.
    """
    if pd.isna(comment):
        return []
    return [segment.strip() for segment in re.split(r"[;,]", str(comment)) if segment.strip()]


def parse_100_tos_comment(comment: Any) -> tuple[list[tuple[str, int]], list[dict[str, str]]]:
    """Return parsed `(code, score)` pairs plus diagnostics for skipped segments."""
    parsed: list[tuple[str, int]] = []
    diagnostics: list[dict[str, str]] = []

    for segment in split_100_tos_comment(comment):
        match = re.match(r"^(?P<code>[A-Za-z_]+)\s+(?P<score>-?1|0)\s*$", segment)
        if match:
            code = canonicalize_hundred_tos_code(match.group("code"))
            score = int(match.group("score"))
            if code in HUNDRED_TOS_CODES:
                parsed.append((code, score))
                continue

        diagnostics.append({
            "segment": segment,
            "reason": classify_unparsed_100_tos_segment(segment),
        })

    return parsed, diagnostics


def derive_100_tos_company(row: dict[str, Any]) -> str | None:
    """Derive company consistently from annotated or cleaned 100 ToS inputs."""
    if "filename" in row and not pd.isna(row["filename"]):
        return str(row["filename"]).removesuffix(".docx")
    for key in ("company", "\ufeffcompany"):
        if key in row and not pd.isna(row[key]):
            return str(row[key])
    return None


def parse_hundred_tos() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Parse 100 ToS comments into long records plus parse diagnostics."""
    df = pd.read_csv(HUNDRED_TOS_CSV_PATH, encoding="utf-8-sig")
    if "referenced_text" not in df.columns:
        raise KeyError("100 ToS input must contain `referenced_text`")

    records: list[dict[str, Any]] = []
    diagnostics: list[dict[str, Any]] = []

    for row_idx, row in enumerate(df.to_dict("records")):
        company = derive_100_tos_company(row)
        source_id = f"{company}:{row.get('comment_id', row_idx)}"
        referenced_text = row.get("referenced_text")
        parsed_pairs, skipped_segments = parse_100_tos_comment(row.get("comment"))

        for skipped in skipped_segments:
            diagnostics.append({
                "row_index": row_idx,
                "source_id": source_id,
                "company": company,
                "comment": row.get("comment"),
                "segment": skipped["segment"],
                "reason": skipped["reason"],
                "referenced_text": referenced_text,
            })

        # Some 100 ToS rows have a valid `code score` comment but no clause text.
        # They cannot become training examples, so log them instead of crashing.
        if not normalize_text(referenced_text):
            for code, score in parsed_pairs:
                diagnostics.append({
                    "row_index": row_idx,
                    "source_id": source_id,
                    "company": company,
                    "comment": row.get("comment"),
                    "segment": f"{code} {score}",
                    "reason": "empty_referenced_text",
                    "referenced_text": referenced_text,
                })
            continue

        for code, score in parsed_pairs:
            topic_ids_for_code = HUNDRED_TOS_CODE_TO_TOPICS[code]
            for topic_id in topic_ids_for_code:
                records.append(
                    make_long_record(
                        text=referenced_text,
                        source_dataset="100_tos",
                        source_id=source_id,
                        company=company,
                        lawgic_topic_id=topic_id,
                        mapped_score=score,
                        native_label=code,
                        native_tag=str(row.get("comment")),
                        native_score=score,
                        mapping_rule=f"100_tos:{code}",
                        metadata={
                            "comment_id": row.get("comment_id"),
                            "author": row.get("author"),
                            "comment": row.get("comment"),
                            "grouping_context": "company+normalized_text_inside_100_tos; global normalized_text_for_cross_source_fusion",
                        },
                    )
                )

    return pd.DataFrame.from_records(records), pd.DataFrame.from_records(diagnostics)


hundred_tos_long, hundred_tos_diagnostics = parse_hundred_tos()
hundred_tos_long.head()

,text,normalized_text,source_dataset,source_id,service_name,platform,company,lawgic_topic_id,topic_index,mapped_score,presence_label,native_label,native_tag,native_score,parse_status,mapping_rule,metadata_json
0,"YouTube is constantly changing and improving the Service. As part of this continual evolution of our digital content and services, we ma...","YouTube is constantly changing and improving the Service. As part of this continual evolution of our digital content and services, we ma...",100_tos,YouTube:3,None,None,YouTube,service_changes,10,0,1.0,serv_chg,serv_chg 0,0,parsed,100_tos:serv_chg,"{""author"": ""Autor"", ""comment"": ""serv_chg 0"", ""comment_id"": 3, ""grouping_context"": ""company+normalized_text_inside_100_tos; global normal..."
1,"operability issues. We’ll also provide you with an opportunity to export your Content using Google Takeout, subject to applicable law an...","operability issues. We’ll also provide you with an opportunity to export your Content using Google Takeout, subject to applicable law an...",100_tos,YouTube:5,None,None,YouTube,transfer_of_contract,23,1,1.0,transfer,tran 1,1,parsed,100_tos:transfer,"{""author"": ""Autor"", ""comment"": ""tran 1"", ""comment_id"": 5, ""grouping_context"": ""company+normalized_text_inside_100_tos; global normalized..."
2,"operability issues. We’ll also provide you with an opportunity to export your Content using Google Takeout, subject to applicable law an...","operability issues. We’ll also provide you with an opportunity to export your Content using Google Takeout, subject to applicable law an...",100_tos,YouTube:5,None,None,YouTube,business_transfer,24,1,1.0,transfer,tran 1,1,parsed,100_tos:transfer,"{""author"": ""Autor"", ""comment"": ""tran 1"", ""comment_id"": 5, ""grouping_context"": ""company+normalized_text_inside_100_tos; global normalized..."
3,"By providing Content to the Service, you grant to YouTube a worldwide, non-exclusive, royalty-free, transferable, sublicensable licence ...","By providing Content to the Service, you grant to YouTube a worldwide, non-exclusive, royalty-free, transferable, sublicensable licence ...",100_tos,YouTube:6,None,None,YouTube,copyright_license,19,0,1.0,IP,ip 0,0,parsed,100_tos:IP,"{""author"": ""Autor"", ""comment"": ""ip 0"", ""comment_id"": 6, ""grouping_context"": ""company+normalized_text_inside_100_tos; global normalized_t..."
4,"By providing Content to the Service, you grant to YouTube a worldwide, non-exclusive, royalty-free, transferable, sublicensable licence ...","By providing Content to the Service, you grant to YouTube a worldwide, non-exclusive, royalty-free, transferable, sublicensable licence ...",100_tos,YouTube:6,None,None,YouTube,ownership,20,0,1.0,IP,ip 0,0,parsed,100_tos:IP,"{""author"": ""Autor"", ""comment"": ""ip 0"", ""comment_id"": 6, ""grouping_context"": ""company+normalized_text_inside_100_tos; global normalized_t..."


## Combine Long Records and Validate

The long table is the audit layer. It should be inspected before any model training.

Validation checks:

- every `lawgic_topic_id` exists in the taxonomy;
- every `mapped_score` is `-1`, `0`, or `1`;
- every row has non-empty normalized text;
- every row represents an explicit source annotation.

The long table may contain multiple rows for the same text and topic because different sources or duplicate annotator comments can agree. That is expected. Conflicts are handled in the next section.

In [18]:
def validate_long_df(long_df: pd.DataFrame) -> None:
    """Validate invariants for the combined long-format dataframe."""
    required_columns = {
        "text",
        "normalized_text",
        "source_dataset",
        "source_id",
        "lawgic_topic_id",
        "topic_index",
        "mapped_score",
        "presence_label",
        "native_label",
        "parse_status",
        "mapping_rule",
        "metadata_json",
    }
    missing_columns = required_columns - set(long_df.columns)
    if missing_columns:
        raise ValueError(f"Long dataframe missing columns: {sorted(missing_columns)}")

    unknown_topics = sorted(set(long_df["lawgic_topic_id"]) - valid_topic_ids)
    if unknown_topics:
        raise ValueError(f"Unknown topic IDs in long dataframe: {unknown_topics}")

    invalid_scores = sorted(set(long_df["mapped_score"]) - {-1, 0, 1})
    if invalid_scores:
        raise ValueError(f"Invalid mapped scores: {invalid_scores}")

    empty_text_rows = long_df[long_df["normalized_text"].astype(str).str.len() == 0]
    if len(empty_text_rows):
        raise ValueError(f"Found {len(empty_text_rows)} rows with empty normalized text")


long_df = pd.concat(
    [claudette_long, tosdr_long, hundred_tos_long],
    ignore_index=True,
    sort=False,
)
validate_long_df(long_df)

long_summary = {
    "rows_by_source": long_df["source_dataset"].value_counts().to_dict(),
    "rows_by_topic": long_df["lawgic_topic_id"].value_counts().to_dict(),
    "rows_by_score": long_df["mapped_score"].value_counts().sort_index().to_dict(),
    "unique_normalized_texts": int(long_df["normalized_text"].nunique()),
    "hundred_tos_diagnostic_reasons": hundred_tos_diagnostics["reason"].value_counts().to_dict()
    if not hundred_tos_diagnostics.empty else {},
    "tosdr_diagnostic_reasons": tosdr_diagnostics["reason"].value_counts().to_dict()
    if not tosdr_diagnostics.empty else {},
}

long_summary

{'rows_by_source': {'tos_dr': 44317, 'claudette': 3721, '100_tos': 2048},
 'rows_by_topic': {'contract_by_use': 3732,
  'complaint_system': 3654,
  'governance': 3362,
  'content_removal': 2749,
  'privacy_incorporation': 2576,
  'trackers': 2524,
  'discretionary_interpretation': 2476,
  'account_termination': 2025,
  'warranty_disclaimer': 1685,
  'personal_data': 1575,
  'contract_changes': 1545,
  'transparency': 1408,
  'recommender_transparency': 1408,
  'interpretation_clause': 1381,
  'account_suspension': 1347,
  'content_rules': 1339,
  'choice_of_law': 1207,
  'content_retrieval': 1206,
  'limitation_of_liability': 1202,
  'choice_of_forum': 1181,
  'third_parties': 1154,
  'notice_of_changes': 979,
  'feedback_reuse': 968,
  'right_to_leave': 795,
  'security': 728,
  'advertising': 663,
  'copyright_license': 628,
  'information_collected': 620,
  'mandatory_arbitration': 575,
  'class_action_waiver': 568,
  'ownership': 518,
  'payments': 353,
  'anonymity': 352,
  'busin

## Detect Score Conflicts

A conflict occurs when the same normalized text and Lawgic topic receives more than one score.

Examples:

- 100 ToS says `limitation_of_liability = 0`, while CLAUDETTE says the same normalized text is `-1`.
- Duplicate 100 ToS rows for the same company/text/topic disagree.

This notebook does not silently collapse conflicts. It writes a review table. For the wide table:

- non-conflicting topics get their single score;
- conflicting topics get `null` in the score vector and appear in `conflict_topic_ids`;
- presence and mask still stay `1`, because all conflicting rows agree that the topic exists.

In [19]:
def build_conflicts_df(long_df: pd.DataFrame) -> pd.DataFrame:
    """Return rows where one text/topic pair has conflicting harm scores."""
    conflict_rows: list[dict[str, Any]] = []

    for (normalized_text, topic_id), group in long_df.groupby(["normalized_text", "lawgic_topic_id"], sort=False):
        scores = sorted(set(int(score) for score in group["mapped_score"]))
        if len(scores) <= 1:
            continue

        conflict_rows.append({
            "normalized_text": normalized_text,
            "display_text": max(group["text"].astype(str), key=len),
            "lawgic_topic_id": topic_id,
            "topic_index": topic_id_to_index[topic_id],
            "scores": compact_json(scores),
            "source_datasets": compact_json(sorted(set(group["source_dataset"].astype(str)))),
            "native_annotations": compact_json(
                group[[
                    "source_dataset",
                    "source_id",
                    "native_label",
                    "native_tag",
                    "mapped_score",
                    "mapping_rule",
                ]].to_dict("records")
            ),
            "num_rows": int(len(group)),
        })

    return pd.DataFrame.from_records(conflict_rows)


conflicts_df = build_conflicts_df(long_df)
conflicts_df.head()

,normalized_text,display_text,lawgic_topic_id,topic_index,scores,source_datasets,native_annotations,num_rows
0,"Amazon reserves the right to refuse service, terminate accounts, terminate your rights to use Amazon Services, remove or edit content, o...","Amazon reserves the right to refuse service, terminate accounts, terminate your rights to use Amazon Services, remove or edit content, o...",content_removal,16,"[-1, 0]","[""claudette"", ""tos_dr""]","[{""mapped_score"": -1, ""mapping_rule"": ""claudette:cr"", ""native_label"": ""cr"", ""native_tag"": ""cr3"", ""source_dataset"": ""claudette"", ""source_...",2
1,"Amazon reserves the right to refuse service, terminate accounts, terminate your rights to use Amazon Services, remove or edit content, o...","Amazon reserves the right to refuse service, terminate accounts, terminate your rights to use Amazon Services, remove or edit content, o...",account_termination,14,"[-1, 0]","[""claudette"", ""tos_dr""]","[{""mapped_score"": -1, ""mapping_rule"": ""claudette:ter"", ""native_label"": ""ter"", ""native_tag"": ""ter3"", ""source_dataset"": ""claudette"", ""sour...",2
2,We may change or update the criteria from time to time without prior notice and at our discretion.,We may change or update the criteria from time to time without prior notice and at our discretion.,contract_changes,9,"[-1, 0]","[""100_tos"", ""claudette""]","[{""mapped_score"": 0, ""mapping_rule"": ""claudette:ch"", ""native_label"": ""ch"", ""native_tag"": ""ch2"", ""source_dataset"": ""claudette"", ""source_i...",2
3,The Terms and any dispute or claim arising out of or in connection with it or its subject matter (including non-contractual disputes or ...,The Terms and any dispute or claim arising out of or in connection with it or its subject matter (including non-contractual disputes or ...,choice_of_law,0,"[-1, 0]","[""100_tos"", ""claudette""]","[{""mapped_score"": 0, ""mapping_rule"": ""claudette:law"", ""native_label"": ""law"", ""native_tag"": ""law2"", ""source_dataset"": ""claudette"", ""sourc...",2
4,"We reserve the right to modify, amend or change the Terms at any time (a “Change”).","We reserve the right to modify, amend or change the Terms at any time (a “Change”).",contract_changes,9,"[-1, 0]","[""100_tos"", ""claudette""]","[{""mapped_score"": 0, ""mapping_rule"": ""claudette:ch"", ""native_label"": ""ch"", ""native_tag"": ""ch2"", ""source_dataset"": ""claudette"", ""source_i...",2


## Build Wide Training Records

The wide table is the model-facing derivative.

Each row represents one normalized clause text. It carries three parallel vectors of length 45:

- `labels_presence`: `1.0` where the topic is explicitly annotated as present, otherwise `0.0`.
- `mask`: `1.0` where the source explicitly annotated that topic, otherwise `0.0`.
- `scores`: `-1`, `0`, or `1` where score is known and non-conflicting; `null` otherwise.

The pair `labels_presence[i] = 0.0` and `mask[i] = 0.0` means **unknown**, not negative. A masked BCE loss must multiply loss by `mask` before reducing.

In [20]:
def choose_display_text(texts: pd.Series) -> str:
    """Choose the most informative original text for display after grouping."""
    unique_texts = [str(text) for text in texts.dropna().unique()]
    if not unique_texts:
        return ""
    return max(unique_texts, key=len)


def build_wide_df(long_df: pd.DataFrame, conflicts_df: pd.DataFrame) -> pd.DataFrame:
    """Convert long records into one masked multi-label row per normalized text."""
    conflict_lookup = {
        (row["normalized_text"], row["lawgic_topic_id"])
        for row in conflicts_df.to_dict("records")
    }

    wide_records: list[dict[str, Any]] = []

    for normalized_text, group in long_df.groupby("normalized_text", sort=False):
        labels_presence = [0.0] * len(topic_ids)
        mask = [0.0] * len(topic_ids)
        scores: list[int | None] = [None] * len(topic_ids)
        topic_scores: dict[str, int | None] = {}
        conflict_topic_ids: list[str] = []

        for topic_id, topic_group in group.groupby("lawgic_topic_id", sort=False):
            topic_index = topic_id_to_index[topic_id]
            labels_presence[topic_index] = 1.0
            mask[topic_index] = 1.0

            score_values = sorted(set(int(score) for score in topic_group["mapped_score"]))
            if (normalized_text, topic_id) in conflict_lookup or len(score_values) > 1:
                scores[topic_index] = None
                topic_scores[topic_id] = None
                conflict_topic_ids.append(topic_id)
            else:
                scores[topic_index] = score_values[0]
                topic_scores[topic_id] = score_values[0]

        native_annotations = group[[
            "source_dataset",
            "source_id",
            "lawgic_topic_id",
            "mapped_score",
            "native_label",
            "native_tag",
            "mapping_rule",
        ]].to_dict("records")

        wide_records.append({
            "text": choose_display_text(group["text"]),
            "normalized_text": normalized_text,
            "sources": compact_json(sorted(set(group["source_dataset"].astype(str)))),
            "labels_presence": compact_json(labels_presence),
            "mask": compact_json(mask),
            "scores": compact_json(scores),
            "topic_scores": compact_json(topic_scores),
            "active_topic_ids": compact_json(sorted(topic_scores.keys(), key=topic_id_to_index.get)),
            "conflict_topic_ids": compact_json(sorted(conflict_topic_ids, key=topic_id_to_index.get)),
            "has_score_conflict": bool(conflict_topic_ids),
            "native_annotations": compact_json(native_annotations),
        })

    return pd.DataFrame.from_records(wide_records)


wide_df = build_wide_df(long_df, conflicts_df)
wide_df.head()

,text,normalized_text,sources,labels_presence,mask,scores,topic_scores,active_topic_ids,conflict_topic_ids,has_score_conflict,native_annotations
0,"By using the Service, you agree to these TOS.","By using the Service, you agree to these TOS.","[""claudette""]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 0, null, null, null, null, null, ...","{""contract_by_use"": 0}","[""contract_by_use""]",[],False,"[{""lawgic_topic_id"": ""contract_by_use"", ""mapped_score"": 0, ""mapping_rule"": ""claudette:use"", ""native_label"": ""use"", ""native_tag"": ""use2"",..."
1,You therefore acknowledge and agree that the form and nature of the Services which 23andMe provides may change from time to time.,You therefore acknowledge and agree that the form and nature of the Services which 23andMe provides may change from time to time.,"[""claudette""]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[null, null, null, null, null, null, null, null, null, 0, null, null, null, null, null, null, null, null, null, null, null, null, null, ...","{""contract_changes"": 0}","[""contract_changes""]",[],False,"[{""lawgic_topic_id"": ""contract_changes"", ""mapped_score"": 0, ""mapping_rule"": ""claudette:ch"", ""native_label"": ""ch"", ""native_tag"": ""ch2"", ""..."
2,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) providing some Servi...","As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) providing some Servi...","[""claudette""]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[null, null, null, null, null, null, null, null, null, 0, null, null, null, null, -1, null, null, null, null, null, null, null, null, nu...","{""account_termination"": -1, ""contract_changes"": 0}","[""contract_changes"", ""account_termination""]",[],False,"[{""lawgic_topic_id"": ""contract_changes"", ""mapped_score"": 0, ""mapping_rule"": ""claudette:ch"", ""native_label"": ""ch"", ""native_tag"": ""ch2"", ""..."
3,You acknowledge and agree that while 23andMe may not currently have set a fixed upper limit on the number of transmissions you may send ...,You acknowledge and agree that while 23andMe may not currently have set a fixed upper limit on the number of transmissions you may send ...,"[""claudette""]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[null, null, null, null, null, null, null, null, null, 0, null, null, null, null, null, null, null, null, null, null, null, null, null, ...","{""contract_changes"": 0}","[""contract_changes""]",[],False,"[{""lawgic_topic_id"": ""contract_changes"", ""mapped_score"": 0, ""mapping_rule"": ""claudette:ch"", ""native_label"": ""ch"", ""native_tag"": ""ch2"", ""..."
4,In case of breach of any one of these promises 23andMe may suspend or terminate your account and refuse any and all current or future us...,In case of breach of any one of these promises 23andMe may suspend

## Write Outputs

Running the next cell writes all derived artifacts under `generated_files/lawgic_taxonomy/`.

Expected outputs:

- `lawgic_combined_long.csv`: full audit table, one row per mapped source annotation.
- `lawgic_combined_wide.csv`: one row per normalized text with `labels_presence`, `mask`, and `scores` vectors.
- `lawgic_combined_conflicts.csv`: text/topic pairs with conflicting harm scores.
- `lawgic_100_tos_parse_diagnostics.csv`: excluded 100 ToS segments and reasons.
- `lawgic_tosdr_parse_diagnostics.csv`: ToS;DR rows skipped because they had empty quote text or unmapped topics.
- `lawgic_fusion_summary.json`: counts and sanity-check metadata.

Do not treat the wide table as final training data until `lawgic_combined_conflicts.csv` has been reviewed.

In [21]:
def build_summary() -> dict[str, Any]:
    """Create a compact machine-readable summary for audit logs."""
    return {
        "taxonomy_path": str(TAXONOMY_PATH.relative_to(PROJECT_ROOT)),
        "num_lawgic_topics": len(topic_ids),
        "topic_ids": topic_ids,
        "input_paths": {
            "claudette": str(CLAUDETTE_CSV_PATH.relative_to(PROJECT_ROOT)),
            "tos_dr": str(TOSDR_CSV_PATH.relative_to(PROJECT_ROOT)),
            "100_tos": str(HUNDRED_TOS_CSV_PATH.relative_to(PROJECT_ROOT)),
            "100_tos_eval_variables": str(HUNDRED_TOS_VARIABLES_PATH.relative_to(PROJECT_ROOT)),
        },
        "output_paths": {
            "long": str(LONG_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
            "wide": str(WIDE_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
            "conflicts": str(CONFLICTS_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
            "100_tos_diagnostics": str(HUNDRED_TOS_DIAGNOSTICS_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
            "tosdr_diagnostics": str(TOSDR_DIAGNOSTICS_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
        },
        "mapping_notes": MAPPING_NOTES,
        "score_policies": {
            "claudette": "Use existing mapped_score; ignore binary_label for harm score.",
            "tos_dr": TOSDR_SCORE_MAP,
            "100_tos": "Use parsed eval-variable score from comment field.",
        },
        "long_summary": long_summary,
        "num_long_rows": int(len(long_df)),
        "num_wide_rows": int(len(wide_df)),
        "num_conflict_rows": int(len(conflicts_df)),
        "num_100_tos_diagnostic_rows": int(len(hundred_tos_diagnostics)),
        "num_tosdr_diagnostic_rows": int(len(tosdr_diagnostics)),
    }


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

long_df.to_csv(LONG_OUTPUT_PATH, index=False)
wide_df.to_csv(WIDE_OUTPUT_PATH, index=False)
conflicts_df.to_csv(CONFLICTS_OUTPUT_PATH, index=False)
hundred_tos_diagnostics.to_csv(HUNDRED_TOS_DIAGNOSTICS_OUTPUT_PATH, index=False)
tosdr_diagnostics.to_csv(TOSDR_DIAGNOSTICS_OUTPUT_PATH, index=False)

with SUMMARY_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(build_summary(), f, ensure_ascii=False, indent=2)

print(f"Wrote long records: {LONG_OUTPUT_PATH}")
print(f"Wrote wide records: {WIDE_OUTPUT_PATH}")
print(f"Wrote conflicts: {CONFLICTS_OUTPUT_PATH}")
print(f"Wrote 100 ToS diagnostics: {HUNDRED_TOS_DIAGNOSTICS_OUTPUT_PATH}")
print(f"Wrote ToS;DR diagnostics: {TOSDR_DIAGNOSTICS_OUTPUT_PATH}")
print(f"Wrote summary: {SUMMARY_OUTPUT_PATH}")

Wrote long records: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/lawgic_combined_long.csv
Wrote wide records: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/lawgic_combined_wide.csv
Wrote conflicts: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/lawgic_combined_conflicts.csv
Wrote 100 ToS diagnostics: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/lawgic_100_tos_parse_diagnostics.csv
Wrote ToS;DR diagnostics: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/lawgic_tosdr_parse_diagnostics.csv
Wrote summary: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/lawgic_fusion_summary.json


## Training Notes: Masked BCE First, Score Head Later

The first model update should use **topic presence** with masked BCE:

```python
loss_per_topic = BCEWithLogitsLoss(reduction="none")(logits, labels_presence)
masked_loss = (loss_per_topic * mask).sum() / mask.sum().clamp_min(1.0)
```

Why masking is mandatory:

- `labels_presence[i] = 0` and `mask[i] = 0` means **unknown**.
- It must not contribute loss.
- Standard BCE without a mask would punish correct predictions for topics that a source never annotated.

Why score prediction is not trained first:

- The existing finetuning notebook is binary multi-label.
- Presence prediction is easier to debug while validating the fused data.
- This notebook still preserves `mapped_score`, `scores`, and `topic_scores`, so a later score head can use the same fusion outputs.

Suggested later architecture:

1. Shared Legal-BERT encoder.
2. Topic-presence head trained with masked BCE.
3. Harm-score head trained only where `mask=1` and non-conflicting score exists, using 3-class cross entropy or an ordinal loss.